# Sweeping a parameter

One scenario per value of `approved.limit`, over a million synthetic records.
`batch_size` sets how many records run at a time: nothing holds the whole
input's results, and the `Progress` callback reports after every batch.

In [1]:
import polars as pl
from decider.experiments import (Bars, Changes, Describe, Divergence, Experiment, Progress, RunStatus,
                                 ValueCounts)
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

income,debt
f64,f64
1000.0,250.0
2000.0,500.0
1500.0,700.0
800.0,250.0


In [2]:
import numpy as np

rng = np.random.default_rng(0)
loans = pl.DataFrame({"income": rng.uniform(500, 5000, 1_000_000), "debt": rng.uniform(0, 2000, 1_000_000)})

exp = Experiment("limit-sweep", pipeline, loans, entry=ENTRY, batch_size=250_000)
for limit in (0.25, 0.3, 0.35, 0.4, 0.45):
    exp.scenario(f"limit_{limit}", params={"credit": {"approved": {"limit": limit}}})
exp.metric(Divergence(column="approved"), ValueCounts(column="approved"))
exp.plot(Changes(metric="divergence"), Bars(metric="value_counts"))
exp.callback(Progress())

result = exp.run()
result.status

limit-sweep: 250,000/1,000,000 records, 0.2s
limit-sweep: 500,000/1,000,000 records, 0.2s
limit-sweep: 750,000/1,000,000 records, 0.3s
limit-sweep: 1,000,000/1,000,000 records, 0.3s


<RunStatus.COMPLETED: 'completed'>

`limit_0.35` is the pipeline's own default, so it changes nothing against the
baseline; every other value moves approvals one way or the other.

In [3]:
{name: s.changed for name, s in result.summaries["divergence"].items()}

{'baseline': 0,
 'limit_0.25': 137350,
 'limit_0.3': 68753,
 'limit_0.35': 0,
 'limit_0.4': 68429,
 'limit_0.45': 132410}

In [4]:
result.plots["changes"].show()
result.plots["bars"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'changed': 0, 'compared': 1000000, 'first': None}, {'scenario': 'limit_0.25', 'changed': 137350, 'compared': 1000000, 'first': 0}, {'scenario': 'limit_0.3', 'changed': 68753, 'compared': 1000000, 'first': 41}, {'scenario': 'limit_0.35', 'changed': 0, 'compared': 1000000, 'first': None}, {'scenario': 'limit_0.4', 'changed': 68429, 'compared': 1000000, 'first': 12}, {'scenario': 'limit_0.45', 'changed': 132410, 'compared': 1000000, 'first': 7}]}, 'mark': {'type': 'bar'}, 'encoding': {'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'changed', 'type': 'quantitative'}, {'field': 'compared', 'type': 'quantitative'}, {'field': 'first', 'title': 'first changed record', 'type': 'quantitative'}], 'x': {'field': 'scenario', 'sort': None, 'type': 'nominal'}, 'y': {'field': 'changed', 'title': 'records changed', 'type': 'quantitative'}}, 'title': 'approved changed vs baseline', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'value': 'False', 'records': 543726}, {'scenario': 'baseline', 'value': 'True', 'records': 456274}, {'scenario': 'limit_0.25', 'value': 'False', 'records': 681076}, {'scenario': 'limit_0.25', 'value': 'True', 'records': 318924}, {'scenario': 'limit_0.3', 'value': 'True', 'records': 387521}, {'scenario': 'limit_0.3', 'value': 'False', 'records': 612479}, {'scenario': 'limit_0.35', 'value': 'False', 'records': 543726}, {'scenario': 'limit_0.35', 'value': 'True', 'records': 456274}, {'scenario': 'limit_0.4', 'value': 'True', 'records': 524703}, {'scenario': 'limit_0.4', 'value': 'False', 'records': 475297}, {'scenario': 'limit_0.45', 'value': 'True', 'records': 588684}, {'scenario': 'limit_0.45', 'value': 'False', 'records': 411316}]}, 'mark': {'type': 'bar'}, 'encoding': {'color': {'field': 'scenario', 'type': 'nominal'}, 'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'value', 'type': 'nominal'}, {'field': 'records', 'type': 'quantitative'}], 'x': {'field': 'value', 'title': 'approved', 'type': 'nominal'}, 'xOffset': {'field': 'scenario', 'type': 'nominal'}, 'y': {'field': 'records', 'type': 'quantitative'}}, 'title': 'approved', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

`result.show()` lays the whole run out: status, a table of scenarios, a table
per metric (one row per scenario), then every plot.

In [5]:
result.show()

### limit-sweep: completed

1000000 records at 3ccba4e2683f

scenario,status,records,revision,error
str,str,i64,null,null
"""baseline""","""completed""",1000000,null,null
"""limit_0.25""","""completed""",1000000,null,null
"""limit_0.3""","""completed""",1000000,null,null
"""limit_0.35""","""completed""",1000000,null,null
"""limit_0.4""","""completed""",1000000,null,null
"""limit_0.45""","""completed""",1000000,null,null


**divergence**

scenario,column,compared,changed,first,samples
str,str,i64,i64,i64,str
"""baseline""","""approved""",1000000,0,null,"""[]"""
"""limit_0.25""","""approved""",1000000,137350,0,"""[{""row"": 0, ""expected"": true, …"
"""limit_0.3""","""approved""",1000000,68753,41,"""[{""row"": 41, ""expected"": true,…"
"""limit_0.35""","""approved""",1000000,0,null,"""[]"""
"""limit_0.4""","""approved""",1000000,68429,12,"""[{""row"": 12, ""expected"": false…"
"""limit_0.45""","""approved""",1000000,132410,7,"""[{""row"": 7, ""expected"": false,…"


**value_counts**

scenario,column,counts.False,counts.True
str,str,i64,i64
"""baseline""","""approved""",543726,456274
"""limit_0.25""","""approved""",681076,318924
"""limit_0.3""","""approved""",612479,387521
"""limit_0.35""","""approved""",543726,456274
"""limit_0.4""","""approved""",475297,524703
"""limit_0.45""","""approved""",411316,588684


Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'changed': 0, 'compared': 1000000, 'first': None}, {'scenario': 'limit_0.25', 'changed': 137350, 'compared': 1000000, 'first': 0}, {'scenario': 'limit_0.3', 'changed': 68753, 'compared': 1000000, 'first': 41}, {'scenario': 'limit_0.35', 'changed': 0, 'compared': 1000000, 'first': None}, {'scenario': 'limit_0.4', 'changed': 68429, 'compared': 1000000, 'first': 12}, {'scenario': 'limit_0.45', 'changed': 132410, 'compared': 1000000, 'first': 7}]}, 'mark': {'type': 'bar'}, 'encoding': {'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'changed', 'type': 'quantitative'}, {'field': 'compared', 'type': 'quantitative'}, {'field': 'first', 'title': 'first changed record', 'type': 'quantitative'}], 'x': {'field': 'scenario', 'sort': None, 'type': 'nominal'}, 'y': {'field': 'changed', 'title': 'records changed', 'type': 'quantitative'}}, 'title': 'approved changed vs baseline', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'value': 'False', 'records': 543726}, {'scenario': 'baseline', 'value': 'True', 'records': 456274}, {'scenario': 'limit_0.25', 'value': 'False', 'records': 681076}, {'scenario': 'limit_0.25', 'value': 'True', 'records': 318924}, {'scenario': 'limit_0.3', 'value': 'True', 'records': 387521}, {'scenario': 'limit_0.3', 'value': 'False', 'records': 612479}, {'scenario': 'limit_0.35', 'value': 'False', 'records': 543726}, {'scenario': 'limit_0.35', 'value': 'True', 'records': 456274}, {'scenario': 'limit_0.4', 'value': 'True', 'records': 524703}, {'scenario': 'limit_0.4', 'value': 'False', 'records': 475297}, {'scenario': 'limit_0.45', 'value': 'True', 'records': 588684}, {'scenario': 'limit_0.45', 'value': 'False', 'records': 411316}]}, 'mark': {'type': 'bar'}, 'encoding': {'color': {'field': 'scenario', 'type': 'nominal'}, 'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'value', 'type': 'nominal'}, {'field': 'records', 'type': 'quantitative'}], 'x': {'field': 'value', 'title': 'approved', 'type': 'nominal'}, 'xOffset': {'field': 'scenario', 'type': 'nominal'}, 'y': {'field': 'records', 'type': 'quantitative'}}, 'title': 'approved', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})